# 02 — Self-Attention from Scratch

In this notebook we build the core attention mechanism that powers every transformer model.

## Table of Contents
1. [What is Attention?](#1-what-is-attention)
2. [Queries, Keys, and Values](#2-queries-keys-and-values)
3. [Scaled Dot-Product Attention](#3-scaled-dot-product-attention)
4. [Step-by-Step Computation](#4-step-by-step-computation)
5. [Why Scaling Matters](#5-why-scaling-matters)
6. [Causal Masking](#6-causal-masking)
7. [Visualising Attention Weights](#7-visualising-attention-weights)
8. [Key Takeaways](#8-key-takeaways)

---

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import sys, os

# Make utils importable from this notebook
sys.path.insert(0, os.path.abspath('.'))
from utils.visualisation import plot_attention_heatmap

# Reproducibility
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. What is Attention?

Attention is a mechanism that lets a model **dynamically focus on the most relevant parts** of its input when producing each output. Instead of treating every input token equally, the model learns to assign different amounts of **attention** (importance) to different tokens.

In a transformer, self-attention means each token in a sequence looks at **all other tokens** (including itself) to gather context. The result is a **contextualised representation** — each token's output encodes information from the tokens it attended to.

### Real-World Analogy

Think of reading a sentence: *"The cat sat on the mat because **it** was tired."*

When you read "it", your brain automatically links it back to "cat" — not "mat". Attention is the mechanism that lets the model make the same connection.

---

## 2. Queries, Keys, and Values

Attention uses three vectors for each token:

| Vector | Role | Analogy |
|--------|------|--------|
| **Query (Q)** | "What am I looking for?" | A search query |
| **Key (K)** | "What do I contain?" | An index/label describing the content |
| **Value (V)** | "What information do I carry?" | The actual content/payload |

The attention mechanism works like a **soft lookup table**:
1. Compare each Query against all Keys (dot product) → similarity scores
2. Convert scores to probabilities (softmax) → attention weights
3. Use weights to take a weighted sum of Values → contextualised output

In self-attention, Q, K, and V are all derived from the **same input** (each token generates its own Q, K, and V through learned linear projections).

In [ ]:
# ============================================================
# Let's start with a tiny example: 4 tokens, embedding dim = 3
# ============================================================

# Imagine these are the embeddings for: ["The", "cat", "sat", "down"]
# Each row is one token's embedding vector (4 tokens × 3 dimensions)
seq_len = 4
d_model = 3

x = torch.tensor([
    [1.0, 0.0, 1.0],   # "The"
    [0.0, 1.0, 1.0],   # "cat"
    [1.0, 1.0, 0.0],   # "sat"
    [0.0, 0.0, 1.0],   # "down"
])  # shape: (seq_len=4, d_model=3)

print(f"Input shape: {x.shape}")
print(f"Input (each row is a token embedding):")
print(x)

In [ ]:
# ============================================================
# In real transformers, Q, K, V are produced by learned linear
# projections: Q = x @ W_q, K = x @ W_k, V = x @ W_v
#
# For now, let's keep it simple and use the raw embeddings
# as Q, K, and V (i.e., W_q = W_k = W_v = Identity).
# This helps us focus purely on the attention computation.
# ============================================================

Q = x.clone()  # (4, 3) — what each token is looking for
K = x.clone()  # (4, 3) — what each token advertises
V = x.clone()  # (4, 3) — what each token carries

print("Q (queries):")
print(Q)
print("\nK (keys):")
print(K)
print("\nV (values):")
print(V)

## 3. Scaled Dot-Product Attention

The attention formula from "Attention Is All You Need" (Vaswani et al., 2017):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:
- $Q$ — query matrix, shape $(\text{seq\_len}, d_k)$
- $K$ — key matrix, shape $(\text{seq\_len}, d_k)$
- $V$ — value matrix, shape $(\text{seq\_len}, d_v)$
- $d_k$ — dimension of keys/queries (used for scaling)
- $QK^T$ — dot product of each query with every key → raw similarity scores
- $\sqrt{d_k}$ — scaling factor to prevent large dot products
- softmax — converts scores to a probability distribution (row-wise)

Let's implement this step by step.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention from scratch.
    
    Args:
        Q: Queries, shape (..., seq_len, d_k)
        K: Keys,    shape (..., seq_len, d_k)
        V: Values,  shape (..., seq_len, d_v)
        mask: Optional boolean mask, True = positions to BLOCK
    
    Returns:
        output:       shape (..., seq_len, d_v)
        attn_weights: shape (..., seq_len, seq_len)
    """
    d_k = Q.size(-1)  # dimension of queries/keys
    
    # Step 1: Compute raw attention scores via dot product
    # Q @ K^T → (seq_len, seq_len)
    # Each element [i, j] measures: "how similar is query i to key j?"
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (..., seq_len, seq_len)
    
    # Step 2: Scale by sqrt(d_k) to prevent large values
    # Large scores → softmax outputs near 0 or 1 → vanishing gradients
    scaled_scores = scores / math.sqrt(d_k)
    
    # Step 3: Apply mask (if provided)
    # Set masked positions to -inf → softmax will give them ~0 weight
    if mask is not None:
        scaled_scores = scaled_scores.masked_fill(mask, float('-inf'))
    
    # Step 4: Softmax over the key dimension (last dim)
    # Each row becomes a probability distribution summing to 1
    attn_weights = F.softmax(scaled_scores, dim=-1)  # (..., seq_len, seq_len)
    
    # Step 5: Weighted sum of values
    # attn_weights @ V → for each query, mix value vectors according to weights
    output = torch.matmul(attn_weights, V)  # (..., seq_len, d_v)
    
    return output, attn_weights

print("Function defined. Let's trace through it step by step.")

## 4. Step-by-Step Computation

Let's trace through each step of the attention computation with our tiny example.

In [ ]:
# ============================================================
# STEP 1: Raw attention scores  QK^T
# ============================================================
# Q @ K^T:  (4, 3) @ (3, 4) → (4, 4)
# Element [i, j] = dot product of query i and key j

scores = torch.matmul(Q, K.transpose(-2, -1))

tokens = ["The", "cat", "sat", "down"]

print("Raw attention scores (QK^T):")
print(scores)
print(f"\nShape: {scores.shape}  — (seq_len × seq_len)")

print("\n" + "="*60)
print("READING THE SCORES MATRIX:")
print("="*60)
for i in range(seq_len):
    print(f"\n  {tokens[i]} (Query {i}) similarity to each key:")
    for j in range(seq_len):
        print(f"    → {tokens[j]} (Key {j}): {scores[i, j]:.1f}")

In [ ]:
# ============================================================
# STEP 2: Scale by sqrt(d_k)
# ============================================================
d_k = Q.size(-1)  # 3

scaled_scores = scores / math.sqrt(d_k)

print(f"d_k = {d_k}")
print(f"sqrt(d_k) = {math.sqrt(d_k):.4f}")
print(f"\nScaled scores (QK^T / sqrt(d_k)):")
print(scaled_scores)
print(f"\nNotice: values are smaller → softer softmax distribution")

In [ ]:
# ============================================================
# STEP 3: Softmax → attention weights (probabilities)
# ============================================================
# Each ROW becomes a probability distribution (sums to 1)
# This tells us: for each query, how much attention to pay to each key

attn_weights = F.softmax(scaled_scores, dim=-1)

print("Attention weights (softmax of scaled scores):")
print(attn_weights)

print("\n" + "="*60)
print("VERIFICATION: Each row sums to 1.0")
print("="*60)
for i in range(seq_len):
    row_sum = attn_weights[i].sum().item()
    print(f"  Row {i} ({tokens[i]}): sum = {row_sum:.6f}")

In [ ]:
# ============================================================
# STEP 4: Weighted sum of values → output
# ============================================================
# attn_weights @ V:  (4, 4) @ (4, 3) → (4, 3)
# For each query, mix value vectors according to attention weights

output = torch.matmul(attn_weights, V)

print("Output (contextualised embeddings):")
print(output)
print(f"\nShape: {output.shape}  — same as input (seq_len × d_model)")

print("\n" + "="*60)
print("HOW EACH OUTPUT IS COMPUTED:")
print("="*60)
for i in range(seq_len):
    w = attn_weights[i].numpy()
    print(f"\n  Output for '{tokens[i]}' (Query {i}):")
    print(f"  = ", end="")
    parts = []
    for j in range(seq_len):
        parts.append(f"{w[j]:.3f} × V_{tokens[j]}")
    print(" + ".join(parts))
    
    # Manual computation
    manual = sum(w[j] * V[j].numpy() for j in range(seq_len))
    print(f"  = {manual}")

In [ ]:
# ============================================================
# Now let's run it all through our function
# ============================================================

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print("Output:")
print(output)
print(f"\nAttention weights:")
print(attn_weights)

In [ ]:
# ============================================================
# Visualise the attention weights as a heatmap
# ============================================================

plot_attention_heatmap(
    attn_weights,
    tokens=tokens,
    title="Self-Attention Weights (No Mask)"
)

### Interpreting the Heatmap

- **Rows** = Queries (the token that's looking)
- **Columns** = Keys (the tokens being looked at)
- **Colour intensity** = how much attention that query pays to that key
- Each row sums to 1.0 (it's a probability distribution)

Notice that each token tends to attend most to itself (diagonal is relatively bright) — this is because the dot product of a vector with itself is maximal.

---

## 5. Why Scaling Matters

The scaling factor $\frac{1}{\sqrt{d_k}}$ is crucial. Without it, as the dimension $d_k$ grows, the dot products grow in magnitude, pushing softmax into regions where its gradients are extremely small.

Let's see this in action.

In [ ]:
# ============================================================
# Demonstrate: What happens without scaling?
# ============================================================

# Create random Q, K, V with increasing d_k
dimensions = [4, 64, 256, 1024]
seq_len_demo = 8

print("="*70)
print("EFFECT OF SCALING ON ATTENTION DISTRIBUTION")
print("="*70)

fig, axes = plt.subplots(2, len(dimensions), figsize=(16, 8))

for idx, d_k in enumerate(dimensions):
    Q_demo = torch.randn(seq_len_demo, d_k)
    K_demo = torch.randn(seq_len_demo, d_k)
    
    # Raw scores (unscaled)
    scores_raw = torch.matmul(Q_demo, K_demo.transpose(-2, -1))
    attn_unscaled = F.softmax(scores_raw, dim=-1)
    
    # Scaled scores
    scores_scaled = scores_raw / math.sqrt(d_k)
    attn_scaled = F.softmax(scores_scaled, dim=-1)
    
    # Print stats
    print(f"\nd_k = {d_k}:")
    print(f"  Unscaled scores — mean: {scores_raw.mean():.2f}, std: {scores_raw.std():.2f}")
    print(f"  Scaled scores   — mean: {scores_scaled.mean():.2f}, std: {scores_scaled.std():.2f}")
    print(f"  Unscaled attn max: {attn_unscaled.max():.4f}, entropy: {-(attn_unscaled * attn_unscaled.log().clamp(min=-100)).sum(-1).mean():.4f}")
    print(f"  Scaled attn max:   {attn_scaled.max():.4f}, entropy: {-(attn_scaled * attn_scaled.log().clamp(min=-100)).sum(-1).mean():.4f}")
    
    # Plot unscaled
    sns.heatmap(attn_unscaled[0:1].numpy(), ax=axes[0, idx], cmap='Blues',
                vmin=0, vmax=1, cbar=idx == len(dimensions)-1)
    axes[0, idx].set_title(f'd_k={d_k} (unscaled)', fontsize=10)
    if idx == 0:
        axes[0, idx].set_ylabel('Unscaled', fontsize=11, fontweight='bold')
    
    # Plot scaled
    sns.heatmap(attn_scaled[0:1].numpy(), ax=axes[1, idx], cmap='Blues',
                vmin=0, vmax=1, cbar=idx == len(dimensions)-1)
    axes[1, idx].set_title(f'd_k={d_k} (scaled)', fontsize=10)
    if idx == 0:
        axes[1, idx].set_ylabel('Scaled', fontsize=11, fontweight='bold')

plt.suptitle('Scaling prevents attention from becoming one-hot as d_k grows',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY INSIGHT:")
print("="*70)
print("Without scaling, as d_k grows, dot products grow too.")
print("Large dot products → softmax becomes nearly one-hot (all weight on one key).")
print("This means: almost no gradient flows to other positions → training breaks.")
print("Dividing by sqrt(d_k) keeps the variance of scores ≈ 1, regardless of d_k.")
print("="*70)

## 6. Causal Masking

For **autoregressive** models (like GPT), each token should only attend to tokens that came **before** it (and itself). It must not "look ahead" at future tokens — that would be cheating during generation.

We achieve this with a **causal mask**: an upper-triangular matrix that blocks attention to future positions.

```
Causal mask for seq_len=4:

         Key₀  Key₁  Key₂  Key₃
Query₀ [  ✓     ✗     ✗     ✗  ]   ← can only see position 0
Query₁ [  ✓     ✓     ✗     ✗  ]   ← can see positions 0-1
Query₂ [  ✓     ✓     ✓     ✗  ]   ← can see positions 0-2
Query₃ [  ✓     ✓     ✓     ✓  ]   ← can see all positions
```

In [ ]:
# ============================================================
# Build a causal mask
# ============================================================
# True = BLOCKED (future positions)
# False = ALLOWED (current and past positions)

causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

print("Causal mask (True = blocked):")
print(causal_mask.int())  # Print as 0/1 for readability
print(f"\nShape: {causal_mask.shape}")

print("\n" + "-"*60)
print("Reading the mask:")
print("-"*60)
for i in range(seq_len):
    allowed = [tokens[j] for j in range(seq_len) if not causal_mask[i, j]]
    print(f"  '{tokens[i]}' (pos {i}) can attend to: {allowed}")

In [ ]:
# ============================================================
# Apply causal mask to attention
# ============================================================

output_causal, attn_weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("Attention weights WITH causal mask:")
print(attn_weights_causal)

print("\n" + "="*60)
print("VERIFICATION: Future positions have zero weight")
print("="*60)
for i in range(seq_len):
    weights = attn_weights_causal[i].numpy()
    print(f"\n  '{tokens[i]}' (Query {i}): {[f'{w:.3f}' for w in weights]}")
    print(f"  Sum = {sum(weights):.6f}")
    for j in range(seq_len):
        if j > i:
            assert weights[j] < 1e-6, f"Future position {j} has non-zero weight!"

print("\n  ✓ All future positions correctly have ~0 weight")

In [ ]:
# ============================================================
# Visualise: Compare masked vs unmasked attention
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Unmasked
plot_attention_heatmap(
    attn_weights, tokens, title="No Mask (bidirectional)", ax=ax1
)

# Causal masked
plot_attention_heatmap(
    attn_weights_causal, tokens, title="Causal Mask (autoregressive)", ax=ax2
)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("KEY INSIGHT:")
print("="*60)
print("With causal masking, the upper triangle is zero.")
print("Each token can only attend to itself and previous tokens.")
print("This is essential for autoregressive generation (GPT-style models)")
print("because at generation time, future tokens don't exist yet.")
print("="*60)

## 7. Visualising Attention Weights

Let's use a more realistic example with actual tokens to see attention patterns.

In [ ]:
# ============================================================
# A more meaningful example using learned projections
# ============================================================
import torch.nn as nn

torch.manual_seed(42)

# Simulate tokens from a real sentence
real_tokens = ["The", "robot", "picked", "up", "the", "box"]
real_seq_len = len(real_tokens)
d_model_real = 16  # small but more realistic embedding dim

# Random embeddings (stand-in for real token embeddings)
x_real = torch.randn(1, real_seq_len, d_model_real)  # (batch=1, seq=6, d=16)

# Learned linear projections (what a real transformer would use)
W_q = nn.Linear(d_model_real, d_model_real, bias=False)
W_k = nn.Linear(d_model_real, d_model_real, bias=False)
W_v = nn.Linear(d_model_real, d_model_real, bias=False)

Q_real = W_q(x_real)  # (1, 6, 16)
K_real = W_k(x_real)  # (1, 6, 16)
V_real = W_v(x_real)  # (1, 6, 16)

# Build causal mask for this sequence
mask_real = torch.triu(
    torch.ones(real_seq_len, real_seq_len, dtype=torch.bool), diagonal=1
)

# Run attention
output_real, attn_real = scaled_dot_product_attention(Q_real, K_real, V_real, mask=mask_real)

# Remove batch dimension for plotting
attn_to_plot = attn_real.squeeze(0)  # (6, 6)

plot_attention_heatmap(
    attn_to_plot,
    tokens=real_tokens,
    title="Attention Weights (Causal, with Learned Projections)",
    figsize=(8, 7)
)

print("\nNote: These patterns are random (untrained weights).")
print("After training, meaningful patterns emerge — e.g., pronouns")
print("attending to their antecedents, verbs attending to subjects, etc.")

In [ ]:
# ============================================================
# Show attention as a bar chart for one specific query
# ============================================================

query_idx = 4  # "the" (second occurrence)
weights_for_query = attn_to_plot[query_idx].detach().numpy()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(real_seq_len), weights_for_query, color='steelblue', alpha=0.8)
ax.set_xticks(range(real_seq_len))
ax.set_xticklabels(real_tokens, fontsize=11)
ax.set_ylabel('Attention Weight', fontsize=11)
ax.set_title(f'Where does "{real_tokens[query_idx]}" (pos {query_idx}) look?',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)

# Annotate values
for bar, w in zip(bars, weights_for_query):
    if w > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{w:.3f}', ha='center', fontsize=9)

# Grey out masked positions
for j in range(query_idx + 1, real_seq_len):
    bars[j].set_color('lightgrey')
    bars[j].set_alpha(0.3)

ax.axhline(y=1/real_seq_len, color='red', linestyle='--', alpha=0.5,
           label=f'Uniform = {1/real_seq_len:.3f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 8. Key Takeaways

---

### Summary

| Concept | What it does |
|---------|-------------|
| **Query (Q)** | Encodes "what am I looking for?" for each token |
| **Key (K)** | Encodes "what do I contain?" — matched against queries |
| **Value (V)** | Carries the actual information to be retrieved |
| **Dot product QK^T** | Measures similarity between each query-key pair |
| **Scaling (÷ √d_k)** | Prevents dot products from growing with dimension → stable softmax |
| **Softmax** | Converts scores to a probability distribution (per query) |
| **Causal mask** | Blocks attention to future positions (for autoregressive models) |
| **Output** | Weighted mix of value vectors — a contextualised representation |

### What's Next?

In the next notebook (**03 — Multi-Head Attention**), we'll see how running **multiple attention heads in parallel** lets the model capture different types of relationships simultaneously.

---